In [1]:
from langchain_openai import OpenAI

In [2]:
import os 
from  dotenv import load_dotenv


In [3]:
import json
import numpy as np              # For numerical operations on embedding vectors
import os
from openai import OpenAI       # OpenAI API client for generating embeddings
from sklearn.metrics.pairwise import cosine_similarity  # Measure similarity between vectors
import matplotlib.pyplot as plt # For visualizing embeddings

load_dotenv()

True

In [9]:
# invoking OpenAI API client

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
# =============================================================================
embedding_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')

In [12]:
#open support tickets 

with open("/workspaces/SupportDesk-RAG-Workshop/data/synthetic_tickets.json", "r") as f:
    support_tickets = json.load(f)

In [13]:
print(f"Loaded {len(support_tickets)} support tickets.")

Loaded 20 support tickets.


In [19]:
print("\n" + "="*80)
print("SAMPLE TICKET:")
print("="*80)
sample = support_tickets[0]
print(f"ID: {sample['ticket_id']}")
print(f"Title: {sample['title']}")
print(f"Description: {sample['description'][:200]}...")
print(f"Category: {sample['category']}")
print(f"Priority: {sample['priority']}")


SAMPLE TICKET:
ID: TICK-001
Title: Users unable to log in after password reset
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment....
Category: Authentication
Priority: High


In [21]:
print("\n" + "="*80)
print("PART 1: Generating Embeddings")
print("="*80)

tiket_texts= [ f"{ticket['title']} {ticket['description']}" for ticket in support_tickets]


PART 1: Generating Embeddings


In [22]:
tiket_texts

["Users unable to log in after password reset Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.",
 "Database connection timeout in production Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.",
 "Payment processing fails for international cards Customers using non-US credit cards receiving 'Payment declined' errors. Payment gateway returns error code 402. Issue only affects cards issued outside United States.",
 'Mobile app crashes on iOS 17 App crashes immediately on launch for users who upgraded to iOS 17. Crash reports show EXC_BAD_ACCESS in networking module. Affects approximately 30% of iOS user base.',
 'Memory leak in background worker process Background job processor showing steady memory increase over time. Process killed by

In [24]:
print("\nGenerating embeddings for all tickets...")
response = client.embeddings.create(input=tiket_texts, model=embedding_model)


Generating embeddings for all tickets...


In [25]:
embeddings = np.array([data.embedding for data in response.data])

In [29]:
embedding_dim = 1536

In [30]:
print(f"✓ Generated embeddings with shape: {embeddings.shape}")
print(f"  ({len(support_tickets)} tickets × {embedding_dim} dimensions)")

✓ Generated embeddings with shape: (20, 1536)
  (20 tickets × 1536 dimensions)


In [31]:
print(f"\nFirst 10 values of embedding for ticket 1:")
print(embeddings[0][:10])
print("  (These 1536 numbers encode the semantic meaning of the text)")


First 10 values of embedding for ticket 1:
[-0.00611877  0.04562378  0.06402588  0.02493286 -0.01404572 -0.02763367
 -0.00489426  0.06365967  0.00029731  0.00773621]
  (These 1536 numbers encode the semantic meaning of the text)


In [32]:
# Create a search query - this is what a user might type
query = "Users can't login after changing password"
print(f"\nSearch Query: '{query}'")


Search Query: 'Users can't login after changing password'


In [33]:
query_response = client.embeddings.create(input=[query], model=embedding_model)
query_embedding = np.array([query_response.data[0].embedding])
print(f"Query embedding shape: {query_embedding.shape}")  # (1, 1536)

Query embedding shape: (1, 1536)


In [34]:
similarities = cosine_similarity(query_embedding, embeddings)[0]
print(f"\nComputed similarity scores for {len(similarities)} tickets")
print(f"Similarity range: [{similarities.min():.4f}, {similarities.max():.4f}]")


Computed similarity scores for 20 tickets
Similarity range: [0.1405, 0.6348]


In [38]:
print("\n" + "="*80)
print("PART 3: Finding Most Similar Tickets")
print("="*80)

# Get top-5 most similar tickets
top_k = 5

# np.argsort() returns indices that would sort the array (ascending)
# [::-1] reverses to get descending order (highest similarity first)
# [:top_k] takes only the top K results
top_indices = np.argsort(similarities)[::-1][:top_k]
print(f"\nTop {top_k} most similar tickets to query: '{query}'")
print("-" * 80)

for rank, idx in enumerate(top_indices, 1):
    ticket = support_tickets[idx]
    score = similarities[idx]
    
    print(f"\n#{rank} - Similarity: {score:.4f}")
    print(f"Ticket ID: {ticket['ticket_id']}")
    print(f"Title: {ticket['title']}")
    print(f"Category: {ticket['category']} | Priority: {ticket['priority']}")
    print(f"Description: {ticket['description'][:150]}...")



PART 3: Finding Most Similar Tickets

Top 5 most similar tickets to query: 'Users can't login after changing password'
--------------------------------------------------------------------------------

#1 - Similarity: 0.6348
Ticket ID: TICK-001
Title: Users unable to log in after password reset
Category: Authentication | Priority: High
Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent sec...

#2 - Similarity: 0.4510
Ticket ID: TICK-011
Title: SSO authentication broken after upgrade
Category: Authentication | Priority: Critical
Description: Single Sign-On integration with corporate SAML provider failing. Users redirected to SSO portal but then receive 'Invalid SAML response' error when re...

#3 - Similarity: 0.4225
Ticket ID: TICK-016
Title: Two-factor authentication codes not working
Category: Authentication | Priority: High
Description: Users unable to log in with TOTP codes